In [ ]:
# 👗 ModeTrends – Fashion Trend Intelligence
## Segmentation Vestimentaire via API Hugging Face

**Auteur :** Jimmy CHABOT  
**Date :**  
**Objectif :**  
Ce notebook présente l'utilisation de l’API Hugging Face pour segmenter automatiquement les vêtements sur des images d’influenceurs mode.

---

## 🎯 Objectifs :
- Interroger une API de segmentation d’images
- Traiter une ou plusieurs images
- Générer un masque de segmentation
- Visualiser les résultats
- Analyser les classes détectées

In [ ]:
# ============================================================
# 1. IMPORTS ET CONFIGURATION
# ============================================================

import os
import requests
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from dotenv import load_dotenv

load_dotenv()
api_token = os.getenv("HF_TOKEN")

if not api_token:
    raise ValueError("❌ Token Hugging Face non trouvé dans .env")

print("✅ Token chargé")

In [ ]:
# ============================================================
# 2. CONFIGURATION API
# ============================================================

API_URL = "https://router.huggingface.co/hf-inference/models/sayeed99/segformer_b3_clothes"

headers = {
    "Authorization": f"Bearer {api_token}"
}

image_path = "chemin/vers/ton/image.jpg"  # Modifier ici

In [ ]:
# ============================================================
# 3. REQUÊTE API
# ============================================================

with open(image_path, "rb") as f:
    image_data = f.read()

response = requests.post(
    API_URL,
    headers={**headers, "Content-Type": "image/jpeg"},
    data=image_data,
    timeout=30
)

response.raise_for_status()
results = response.json()

print(f"✅ {len(results)} classes détectées")

In [ ]:
# ============================================================
# 4. CRÉATION DU MASQUE
# ============================================================

def create_mask(results, width, height):
    mask = np.zeros((height, width))
    
    for i, item in enumerate(results):
        if item["label"] != "Background":
            for coord in item["mask"]:
                mask[coord[1], coord[0]] = i + 1
                
    return mask

original = Image.open(image_path)
width, height = original.size

mask = create_mask(results, width, height)

In [ ]:
# ============================================================
# 5. VISUALISATION
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(original)
axes[0].set_title("Image Originale")
axes[0].axis("off")

axes[1].imshow(mask, cmap="tab20")
axes[1].set_title("Segmentation des vêtements")
axes[1].axis("off")

plt.show()

In [ ]:
# ============================================================
# 6. ANALYSE DES RÉSULTATS
# ============================================================

detected_classes = set([r["label"] for r in results if r["label"] != "Background"])

print("👔 Classes détectées :")
for c in sorted(detected_classes):
    print("-", c)

# 📊 Conclusion

L’API Hugging Face permet une segmentation efficace des vêtements sur des images réelles.

### Points forts :
- Rapidité
- Précision des classes
- Facilité d’intégration

### Limites :
- Dépendance à une API externe
- Temps de réponse variable
- Risque de timeout

### Perspectives :
- Analyse statistique des tendances vestimentaires
- Détection automatique des couleurs dominantes
- Intégration dans une application web